In [11]:
# Importing necessary libraries for EDA
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn



In [4]:
# Step 1: Load and Understand the Dataset

import pandas as pd

# Load the dataset (replace 'your_file.csv' with the actual file path)
df = pd.read_csv('data/data.csv')

# Inspect the structure of the dataset
print("Dataset Shape:", df.shape)
print("\nDataset Info:")
df.info()

# Display summary statistics
print("\nSummary Statistics:")
print(df.describe())

# Display first few rows of the dataset
print("\nFirst Few Rows:")
print(df.head())


Dataset Shape: (95662, 16)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95662 entries, 0 to 95661
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   TransactionId         95662 non-null  object 
 1   BatchId               95662 non-null  object 
 2   AccountId             95662 non-null  object 
 3   SubscriptionId        95662 non-null  object 
 4   CustomerId            95662 non-null  object 
 5   CurrencyCode          95662 non-null  object 
 6   CountryCode           95662 non-null  int64  
 7   ProviderId            95662 non-null  object 
 8   ProductId             95662 non-null  object 
 9   ProductCategory       95662 non-null  object 
 10  ChannelId             95662 non-null  object 
 11  Amount                95662 non-null  float64
 12  Value                 95662 non-null  int64  
 13  TransactionStartTime  95662 non-null  object 
 14  PricingStrategy       95662 

In [5]:
# Step 2: Create Aggregate Features

# Total Transaction Amount per Customer
df['TotalTransactionAmount'] = df.groupby('CustomerId')['Amount'].transform('sum')

# Average Transaction Amount per Customer
df['AverageTransactionAmount'] = df.groupby('CustomerId')['Amount'].transform('mean')

# Transaction Count per Customer
df['TransactionCount'] = df.groupby('CustomerId')['TransactionId'].transform('count')

# Standard Deviation of Transaction Amounts per Customer
df['StdTransactionAmount'] = df.groupby('CustomerId')['Amount'].transform('std')

# Verify the new features
print("Aggregate Features Preview:")
print(df[['CustomerId', 'TotalTransactionAmount', 'AverageTransactionAmount', 'TransactionCount', 'StdTransactionAmount']].head())


Aggregate Features Preview:
        CustomerId  TotalTransactionAmount  AverageTransactionAmount  \
0  CustomerId_4406               109921.75                923.712185   
1  CustomerId_4406               109921.75                923.712185   
2  CustomerId_4683                 1000.00                500.000000   
3   CustomerId_988               228727.20               6019.136842   
4   CustomerId_988               228727.20               6019.136842   

   TransactionCount  StdTransactionAmount  
0               119           3042.294251  
1               119           3042.294251  
2                 2              0.000000  
3                38          17169.241610  
4                38          17169.241610  


In [7]:
# Step 2: Extract Time-Based Features

# Convert TransactionStartTime to datetime format
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

# Extract Transaction Hour
df['TransactionHour'] = df['TransactionStartTime'].dt.hour

# Extract Transaction Day
df['TransactionDay'] = df['TransactionStartTime'].dt.day

# Extract Transaction Month
df['TransactionMonth'] = df['TransactionStartTime'].dt.month

# Extract Transaction Year
df['TransactionYear'] = df['TransactionStartTime'].dt.year

# Verify the new features
print("Time-Based Features Preview:")
print(df[['TransactionStartTime', 'TransactionHour', 'TransactionDay', 'TransactionMonth', 'TransactionYear']].head())


Time-Based Features Preview:
       TransactionStartTime  TransactionHour  TransactionDay  \
0 2018-11-15 02:18:49+00:00                2              15   
1 2018-11-15 02:19:08+00:00                2              15   
2 2018-11-15 02:44:21+00:00                2              15   
3 2018-11-15 03:32:55+00:00                3              15   
4 2018-11-15 03:34:21+00:00                3              15   

   TransactionMonth  TransactionYear  
0                11             2018  
1                11             2018  
2                11             2018  
3                11             2018  
4                11             2018  


In [10]:
# Step 3: Encode Categorical Variables

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Apply Label Encoding to columns with many unique values
label_encoder = LabelEncoder()
df['CustomerId_encoded'] = label_encoder.fit_transform(df['CustomerId'])
df['ProductId_encoded'] = label_encoder.fit_transform(df['ProductId'])

# Apply One-Hot Encoding to columns with fewer unique values
one_hot_columns = ['CurrencyCode', 'ProductCategory', 'ChannelId']
df = pd.get_dummies(df, columns=one_hot_columns, drop_first=True)

# Verify the encoded features
print("Encoded Features Preview:")
print(df.head())


Encoded Features Preview:
         TransactionId         BatchId       AccountId       SubscriptionId  \
0  TransactionId_76871   BatchId_36123  AccountId_3957   SubscriptionId_887   
1  TransactionId_73770   BatchId_15642  AccountId_4841  SubscriptionId_3829   
2  TransactionId_26203   BatchId_53941  AccountId_4229   SubscriptionId_222   
3    TransactionId_380  BatchId_102363   AccountId_648  SubscriptionId_2185   
4  TransactionId_28195   BatchId_38780  AccountId_4841  SubscriptionId_3829   

        CustomerId  CountryCode    ProviderId     ProductId   Amount  Value  \
0  CustomerId_4406          256  ProviderId_6  ProductId_10   1000.0   1000   
1  CustomerId_4406          256  ProviderId_4   ProductId_6    -20.0     20   
2  CustomerId_4683          256  ProviderId_6   ProductId_1    500.0    500   
3   CustomerId_988          256  ProviderId_1  ProductId_21  20000.0  21800   
4   CustomerId_988          256  ProviderId_4   ProductId_6   -644.0    644   

   ... ProductCategory_f